# Carga no Qdrant

Pega os chunks com seus metadados, gera os embeddings e sobe para o Qdrant.
**Só isso.** Não tem busca, não tem fusão, não tem teste de recuperação.

## Por que este notebook existe

O notebook anterior estourava a memória nesta etapa. A causa não era o batch:

O `token_count` gravado no JSONL foi contado com **tiktoken** (tokenizer da
OpenAI), que neste corpus é **2,55x menor** que o tokenizer do Qwen.

| | Declarado (tiktoken) | Real (Qwen) |
|---|---|---|
| Mediana | 639 | 1.308 |
| **Máximo** | **2.399** | **22.935** |

E o Qwen3-Embedding-0.6B não traz `sentence_bert_config.json`, então o
sentence-transformers assume `max_seq_length = 32768` e não trunca nada. Pior:
ele ordena a entrada por tamanho e processa os **maiores primeiro**, de modo que
a primeira batelada já pegava os 8 maiores chunks, com cerca de 22 mil tokens
cada.

A matriz de atenção é `batch × cabeças × seq²`. Com `batch=8` e `seq=22935` isso
dá cerca de **269 GB por camada**. Por isso o crash foi imediato.

**A correção:** reparticionar os chunks pelo tokenizer real antes de vetorizar,
com teto de 1.024 tokens. A memória cai para 0,27 GB por camada.

## Entrada e saída

| | |
|---|---|
| Entrada | `chunks_fatec_rag.jsonl`, 438 chunks |
| Saída | coleção `vaar_rag` no Qdrant Cloud |

## 1. Dependências

In [ ]:
import importlib.util
import subprocess
import sys

PACOTES = {
    "sentence_transformers": "sentence-transformers>=3.0",
    "transformers": "transformers>=4.51",
    "qdrant_client": "qdrant-client>=1.12",
    "torch": "torch>=2.1",
}
faltando = [p for m, p in PACOTES.items() if not importlib.util.find_spec(m)]
if faltando:
    print("instalando:", ", ".join(faltando))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)
    print("instalado. Se a proxima celula reclamar de import, reinicie o ambiente.")
else:
    print("dependencias ja presentes")

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("   ", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
else:
    print("    SEM GPU: vai levar varios minutos. Colab: Ambiente de execucao > GPU")

## 2. Configuração

In [ ]:
import os
from pathlib import Path

MODELO   = "Qwen/Qwen3-Embedding-0.6B"
DIMENSAO = 1024
COLECAO  = "vaar_rag"

# ── Limites que evitam o estouro de memoria ──────────────────────────────────
# Teto medido no tokenizer REAL do Qwen, nao no token_count do JSONL.
MAX_TOKENS  = 1024      # teto por chunk
OVERLAP     = 128       # sobreposicao ao reparticionar
MIN_TOKENS  = 40        # descarta cabecalho e rodape de pagina
LOTE        = 8         # suba para 16 ou 32 se a GPU tiver folga

# O vetor esparso custa milissegundos de CPU e e o que permite achar
# "art. 14" e "Portaria 14/2025". Deixe True, senao sera preciso reenviar
# todos os pontos depois para adiciona-lo.
INCLUIR_ESPARSO = True

# ── Ambiente ─────────────────────────────────────────────────────────────────
try:
    import google.colab          # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

NOME_CHUNKS = "chunks_fatec_rag.jsonl"
REPO_GIT = "https://github.com/ProjetoIntegrador-V/rag-vaar-nacional.git"

RAIZ = Path.cwd()
if RAIZ.name == "notebooks" and (RAIZ.parent / "scripts").exists():
    RAIZ = RAIZ.parent

# ── Credenciais: nunca escritas aqui ─────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(RAIZ / ".env")
except ImportError:
    pass

QDRANT_URL = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
if not QDRANT_URL:
    QDRANT_URL = input("Cluster endpoint (https://....cloud.qdrant.io): ").strip()
if not QDRANT_API_KEY:
    from getpass import getpass
    QDRANT_API_KEY = getpass("API key do Qdrant (nao aparece na tela): ").strip()

print("ambiente:", "Google Colab" if EM_COLAB else "local")
print("endpoint:", QDRANT_URL)
print("api key :", f"{QDRANT_API_KEY[:10]}...{QDRANT_API_KEY[-4:]}")

## 3. Carregar os chunks

In [ ]:
import json
import subprocess
from collections import Counter


def procurar(raiz):
    for c in [raiz / "data" / "chunks" / NOME_CHUNKS,
              raiz / "scripts" / NOME_CHUNKS,
              Path.cwd() / NOME_CHUNKS,
              Path("/content") / NOME_CHUNKS]:
        if c.exists():
            return c
    return None


CAMINHO = procurar(RAIZ)

if CAMINHO is None and EM_COLAB:
    destino = Path("/content/rag-vaar-nacional")
    if not destino.exists():
        print("clonando o repositorio...")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_GIT, str(destino)],
                           capture_output=True, text=True)
        print("clone ok" if r.returncode == 0 else f"falhou: {r.stderr[:200]}")
    if destino.exists():
        RAIZ = destino
        CAMINHO = procurar(RAIZ)

if CAMINHO is None and EM_COLAB:
    print(f"Envie o arquivo {NOME_CHUNKS}:")
    from google.colab import files
    for nome in files.upload():
        if nome.endswith(".jsonl"):
            CAMINHO = Path.cwd() / nome

if CAMINHO is None:
    raise FileNotFoundError(
        NOME_CHUNKS + " nao encontrado. Procurei em data/chunks/, scripts/, "
        + str(Path.cwd()) + " e /content."
    )

chunks = [json.loads(l) for l in CAMINHO.open(encoding="utf-8") if l.strip()]
print("arquivo :", CAMINHO)
print("chunks  :", len(chunks))
print("docs    :", len({c["document_id"] for c in chunks}))
print("metadados:", sorted({k for c in chunks for k in c["metadata"]}))

## 4. Reparticionar pelos tokens reais

Esta é a célula que corrige o estouro de memória.

Ela mede cada chunk com o **tokenizer do próprio Qwen**, e não com o
`token_count` gravado no JSONL. Chunk acima de 1.024 tokens é cortado em pedaços
com sobreposição, e cada pedaço **herda todos os metadados** do original, com o
`chunk_id` ganhando um sufixo `_s0`, `_s1` e assim por diante.

Reparticionar preserva o conteúdo. Truncar descartaria 95% dos chunks grandes,
que são justamente as tabelas da Portaria.

In [ ]:
from transformers import AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained(MODELO)


def reparticionar(chunk):
    ids = tokenizador(chunk["text"], add_special_tokens=False)["input_ids"]
    if len(ids) <= MAX_TOKENS:
        return [dict(chunk, tokens_qwen=len(ids))]
    saida, passo, i, n = [], MAX_TOKENS - OVERLAP, 0, 0
    while i < len(ids):
        pedaco = ids[i:i + MAX_TOKENS]
        if len(pedaco) >= MIN_TOKENS:
            saida.append(dict(chunk,
                              text=tokenizador.decode(pedaco),
                              chunk_id=f"{chunk['chunk_id']}_s{n}",
                              tokens_qwen=len(pedaco)))
            n += 1
        i += passo
    return saida


descartados = [c for c in chunks if c["token_count"] < 30]
uteis = [c for c in chunks if c["token_count"] >= 30]

finais = []
for c in uteis:
    finais.extend(reparticionar(c))

t = [c["tokens_qwen"] for c in finais]
print(f"descartados (cabecalho/rodape): {len(descartados)}")
print(f"chunks uteis                  : {len(uteis)}")
print(f"apos reparticionar            : {len(finais)}  (+{len(finais) - len(uteis)})")
print(f"tokens reais: min {min(t)} | max {max(t)}")

assert max(t) <= MAX_TOKENS, "sobrou chunk acima do teto"
assert len({c['chunk_id'] for c in finais}) == len(finais), "chunk_id duplicado"
print("\nteto respeitado e ids unicos")

## 5. Embeddings

In [ ]:
import time

import numpy as np
from sentence_transformers import SentenceTransformer

usar_gpu = torch.cuda.is_available()
modelo = SentenceTransformer(
    MODELO,
    model_kwargs={"torch_dtype": torch.float16} if usar_gpu else {},
    tokenizer_kwargs={"padding_side": "left"},   # pooling e de ultimo token
)
# Trava explicita: sem isso o sentence-transformers assume 32768, porque este
# modelo nao traz sentence_bert_config.json.
modelo.max_seq_length = MAX_TOKENS
print(f"modelo carregado | max_seq_length = {modelo.max_seq_length} | "
      f"{'fp16 na GPU' if usar_gpu else 'fp32 na CPU'}")

textos = [c["text"] for c in finais]
inicio = time.perf_counter()
vetores = modelo.encode(
    textos,
    batch_size=LOTE,
    normalize_embeddings=True,     # norma 1: produto interno vira cosseno
    convert_to_numpy=True,
    show_progress_bar=True,
)
vetores = np.asarray(vetores, dtype=np.float32)
print(f"\nmatriz {vetores.shape} em {time.perf_counter() - inicio:.0f}s")

assert vetores.shape == (len(finais), DIMENSAO)
assert np.allclose(np.linalg.norm(vetores, axis=1), 1.0, atol=1e-3), "nao normalizado"
print("dimensao e normas conferidas")

## 6. Vetor esparso

In [ ]:
import re
import unicodedata

from qdrant_client import models

STOPWORDS_PT = {
    "a", "ao", "aos", "as", "da", "das", "de", "do", "dos", "e", "em", "na",
    "nas", "no", "nos", "o", "os", "ou", "para", "pela", "pelas", "pelo",
    "pelos", "por", "que", "se", "um", "uma", "umas", "uns", "com", "como",
    "sao", "foi", "ser", "sua", "seu", "suas", "seus", "isso", "este", "esta",
    "esse", "essa", "aquele", "aquela", "qual", "quais", "quando", "onde",
}
# Aceita ponto, hifen e barra DENTRO do token, para nao picar "art.14",
# "3106200-1" nem "2025/2026", que sao as ancoras que o lado lexical existe
# para acertar.
PADRAO = re.compile(r"[a-z0-9]+(?:[.\-/][a-z0-9]+)*")
vocabulario = {}


def tokenizar(texto):
    sem_ac = "".join(c for c in unicodedata.normalize("NFKD", texto.lower())
                     if not unicodedata.combining(c))
    return [t for t in PADRAO.findall(sem_ac)
            if (len(t) > 1 or t.isdigit()) and (t not in STOPWORDS_PT or t.isdigit())]


def esparso(texto):
    cont = {}
    for tk in tokenizar(texto):
        i = vocabulario.setdefault(tk, len(vocabulario))
        cont[i] = cont.get(i, 0) + 1
    if not cont:
        return None
    return models.SparseVector(indices=list(cont),
                               values=[float(v) for v in cont.values()])


if INCLUIR_ESPARSO:
    esparsos = [esparso(c["text"]) for c in finais]
    print(f"vocabulario: {len(vocabulario)} termos")
    # O Qdrant indexa por indice inteiro, nao por palavra. Sem este mapa a
    # busca esparsa nao casa nada depois.
    destino_vocab = RAIZ / "data" / "vocabulario_esparso.json"
    destino_vocab.parent.mkdir(parents=True, exist_ok=True)
    destino_vocab.write_text(json.dumps(vocabulario, ensure_ascii=False),
                             encoding="utf-8")
    print("vocabulario salvo em", destino_vocab)
    print("GUARDE ESTE ARQUIVO: sem ele a busca esparsa devolve zero resultados.")
else:
    esparsos = [None] * len(finais)
    print("vetor esparso desativado")

## 7. Criar a coleção e subir

In [ ]:
import uuid

from qdrant_client import QdrantClient

cliente = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)
print("colecoes no cluster:", [c.name for c in cliente.get_collections().collections] or "(nenhuma)")

RECRIAR = True
if RECRIAR and cliente.collection_exists(COLECAO):
    cliente.delete_collection(COLECAO)
    print(f"colecao '{COLECAO}' anterior removida")

if not cliente.collection_exists(COLECAO):
    cliente.create_collection(
        collection_name=COLECAO,
        vectors_config={"denso": models.VectorParams(
            size=DIMENSAO, distance=models.Distance.COSINE)},
        sparse_vectors_config=(
            # Modifier.IDF: o Qdrant calcula a raridade do termo. O cliente
            # envia so a frequencia.
            {"esparso": models.SparseVectorParams(modifier=models.Modifier.IDF)}
            if INCLUIR_ESPARSO else {}
        ),
    )
    print(f"colecao '{COLECAO}' criada")

for campo, tipo in [("ano", models.PayloadSchemaType.INTEGER),
                    ("tipo_documento", models.PayloadSchemaType.KEYWORD),
                    ("orgao", models.PayloadSchemaType.KEYWORD),
                    ("document_id", models.PayloadSchemaType.KEYWORD)]:
    try:
        cliente.create_payload_index(COLECAO, field_name=campo, field_schema=tipo)
    except Exception:
        pass
print("indices de payload criados")

In [ ]:
# UUID5 do chunk_id: reexecutar atualiza os mesmos pontos em vez de duplicar.
NAMESPACE = uuid.UUID("6f1a0b3c-6d2e-4a58-9b7f-2c9d5e8a1b40")

pontos = []
for chunk, denso, esp in zip(finais, vetores, esparsos):
    vetor = {"denso": denso.tolist()}
    if INCLUIR_ESPARSO and esp is not None:
        vetor["esparso"] = esp
    meta = chunk.get("metadata", {})
    pontos.append(models.PointStruct(
        id=str(uuid.uuid5(NAMESPACE, chunk["chunk_id"])),
        vector=vetor,
        payload={
            "titulo": chunk["title"],
            "texto": chunk["text"],
            "ano": meta.get("ano"),
            "tipo_documento": meta.get("tipo_documento"),
            "orgao": meta.get("orgao"),
            "document_id": chunk["document_id"],
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "tokens_qwen": chunk["tokens_qwen"],
        },
    ))

TAM = 128
for i in range(0, len(pontos), TAM):
    cliente.upsert(collection_name=COLECAO, points=pontos[i:i + TAM], wait=True)
    print(f"  {min(i + TAM, len(pontos))}/{len(pontos)}", end="\r")

total = cliente.count(COLECAO).count
print(f"\n\ncolecao '{COLECAO}': {total} pontos")
assert total == len(pontos), f"esperado {len(pontos)}, veio {total}"

## 8. Conferência

In [ ]:
info = cliente.get_collection(COLECAO)
print("pontos          :", info.points_count)
print("vetores densos  :", {k: (v.size, v.distance) for k, v in
                            info.config.params.vectors.items()})
if INCLUIR_ESPARSO:
    print("vetores esparsos:", list((info.config.params.sparse_vectors or {}).keys()))

amostra = cliente.scroll(COLECAO, limit=2, with_payload=True, with_vectors=False)[0]
print("\ncampos gravados no payload:", sorted(amostra[0].payload))
print("\nexemplo:")
p = amostra[0].payload
for k in ("titulo", "ano", "tipo_documento", "page", "tokens_qwen", "chunk_id"):
    print(f"  {k:15s} = {str(p.get(k))[:64]}")
print(f"  {'texto':15s} = {p['texto'][:90]}...")

## Pronto

A coleção está no ar com os chunks, os metadados e os vetores.

**Guarde o `data/vocabulario_esparso.json`.** O Qdrant indexa o vetor esparso
por índice inteiro, não por palavra, e esse mapa mora no cliente. A coleção é
compartilhada, o vocabulário não. Quem consultar sem ele recebe zero resultados
no lado lexical, sem erro nenhum.

**Se quiser subir de novo**, é só reexecutar. Os ids são derivados do
`chunk_id`, então os pontos são atualizados, não duplicados.

### Sobre o `token_count` do JSONL

Ele continua errado para este modelo, porque foi contado com tiktoken. Este
notebook contorna reparticionando pelo tokenizer real, mas a correção de fundo
é o `scripts/chunking.py` passar a contar com o tokenizer do modelo que vai
gerar os embeddings. Enquanto isso não acontecer, qualquer outro notebook que
confie no `token_count` vai estourar do mesmo jeito.